In [ ]:
%%configure -f
{
    "vCores": 8
}

## Build the gold layer: DuckDB on a Fabric Python notebook

Phase 2 of the pipeline, running on capacity. The mirrored PostgreSQL table is
read through the shortcut at `Tables/raw_txn`, cleaned, conformed, scored,
deduplicated and shaped into a star schema by DuckDB, then written as parquet
under `Files/gold`. `02_vorder_write` turns that parquet into V-Ordered Delta.

This is a **Python** notebook on a single node, deliberately. At 8 vCores it
bills 4 CU. The same work on a Spark starter pool would hold at least 8 CU.

The first cell sets the node size and must stay first: `%%configure` only takes
effect at session start, and an API-triggered run honours it.

In [ ]:
import os, sys, json, time, shutil, platform

def ram_gb():
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    except Exception:
        return None

ENV = {
    "cpu_count": os.cpu_count(),
    "ram_gb": round(ram_gb() or 0, 1),
    "tmp_free_gb": round(shutil.disk_usage("/tmp").free / 1e9, 1),
    "python": platform.python_version(),
    # A Spark-kernel session pre-defines `spark`; a Python-kernel session does
    # not. Recording it settles which compute this actually ran on.
    "spark_in_globals": "spark" in globals(),
}
print(json.dumps(ENV, indent=2))

### The transformation code, embedded

`transform.py`, `rules.py` and `star.py` are written out and imported here.
They are the exact files validated against the full 50,000,000 rows, embedded
at generation time by `scripts/build_gold_notebook.py`, so the notebook cannot
drift from the code in the repository.

In [ ]:
SOURCES = {
 "transform.py": "\"\"\"\nPhase 2 transformation: raw landing table to conformed silver, in DuckDB SQL.\n\n  python transform.py --source postgres --limit 2000000   # test locally\n  python transform.py --source delta --path /lakehouse/default/Tables/raw_txn\n\nThe SQL here is the same text that runs in the Fabric Python notebook. Keeping it\nin a module means it can be exercised against the real 50,000,000 rows on a\nlaptop before it ever runs on capacity, which is a great deal cheaper than\ndebugging inside a notebook session.\n\nWhat it does, in order\n----------------------\n1. CLEAN    null-lookalikes to real NULL, trim, parse five date formats, parse\n            money written four different ways, normalise casing.\n2. CONFORM  32 spellings of one country to one code, 35 txn_type values to a\n            small set, 11 KYC variants to 5 real statuses, MCC to category.\n3. DEDUPE   593,209 exact duplicate rows, resolved by _mirror_row_id.\n4. DERIVE   the nine risk rules from APP_DESIGN.md and a weighted risk_score.\n\nNothing here invents a correlation. The rules are computed from columns that\ngenuinely vary, and the score is a function of those rules, so it has a real\ndistribution rather than the flat line the source fraud_score column produces.\n\"\"\"\n\nimport argparse\nimport sys\nimport time\n\nimport duckdb\n\nNULLISH = \"('','NULL','N/A','NA','-','unknown','none','#N/A','null','n/a')\"\n\n# --------------------------------------------------------------------------\n# reusable scalar cleaners\n# --------------------------------------------------------------------------\n\ndef s(col):\n    \"\"\"Trim, collapse inner runs of whitespace, and map null-lookalikes to NULL.\"\"\"\n    return (\"nullif(nullif(regexp_replace(trim(coalesce({c},'')), '\\\\s+', ' ', 'g'), ''), \"\n            \"'\\\\x00')\".format(c=col))\n\n\ndef clean_text(col):\n    return (\"CASE WHEN lower(trim(coalesce({c},''))) IN {n} THEN NULL \"\n            \"ELSE {t} END\").format(c=col, n=NULLISH, t=s(col))\n\n\ndef clean_upper(col):\n    return \"upper({0})\".format(clean_text(col))\n\n\ndef money(col):\n    \"\"\"\n    Amounts arrive as 1234.50, '1,234.50', 'RM99.00' and '(45.00)' for credits.\n    Strip the currency word and separators, then honour the accounting negative.\n    \"\"\"\n    body = (\"regexp_replace(regexp_replace(trim(coalesce({c},'')), \"\n            \"'(?i)^(rm|myr|usd|sgd|eur|gbp|aud|jpy)\\\\s*', '', 'g'), '[,\\\\s]', '', 'g')\").format(c=col)\n    return (\"CASE WHEN lower(trim(coalesce({c},''))) IN {n} THEN NULL \"\n            \"WHEN {b} LIKE '(%)' THEN -1 * TRY_CAST(replace(replace({b},'(',''),')','') AS DECIMAL(18,2)) \"\n            \"ELSE TRY_CAST({b} AS DECIMAL(18,2)) END\").format(c=col, n=NULLISH, b=body)\n\n\ndef to_date(col):\n    \"\"\"\n    Five competing formats plus sentinel values. try_strptime returns NULL on a\n    miss, so the coalesce chain walks them in order of frequency.\n    \"\"\"\n    fmts = [\"%Y-%m-%d\", \"%d/%m/%Y\", \"%m/%d/%Y\", \"%-d-%b-%y\", \"%d.%m.%Y\"]\n    chain = \", \".join(\"try_strptime({c}, '{f}')\".format(c=s(col), f=f) for f in fmts)\n    return (\"CASE WHEN trim(coalesce({c},'')) IN ('1900-01-01','9999-12-31','0000-00-00','1970-01-01') \"\n            \"THEN NULL WHEN lower(trim(coalesce({c},''))) IN {n} THEN NULL \"\n            \"ELSE CAST(coalesce({ch}) AS DATE) END\").format(c=col, n=NULLISH, ch=chain)\n\n\ndef to_ts(col):\n    fmts = [\"%Y-%m-%d %H:%M:%S\", \"%Y-%m-%dT%H:%M:%S\", \"%d/%m/%Y %H:%M\", \"%Y-%m-%d %H:%M:%S.%g\"]\n    chain = \", \".join(\"try_strptime({c}, '{f}')\".format(c=s(col), f=f) for f in fmts)\n    # A bare epoch integer also appears in this column. It must become a naive\n    # TIMESTAMP via make_timestamp, not to_timestamp: to_timestamp returns\n    # TIMESTAMPTZ, which coerces the whole CASE to TIMESTAMPTZ, and every hour\n    # and date derived from it then depends on the session's TimeZone. That\n    # made the laptop (Asia/Dhaka) and Fabric (UTC) builds disagree on epoch\n    # rows by six hours. BUILD_LOG section 40.\n    return (\"CASE WHEN lower(trim(coalesce({c},''))) IN {n} THEN NULL \"\n            \"WHEN regexp_matches(trim(coalesce({c},'')), '^[0-9]{{9,11}}$') \"\n            \"THEN make_timestamp(TRY_CAST(trim({c}) AS BIGINT) * 1000000::BIGINT) \"\n            \"ELSE coalesce({ch}, try_strptime(substr({t},1,19), '%Y-%m-%dT%H:%M:%S')) END\"\n            ).format(c=col, n=NULLISH, ch=chain, t=s(col))\n\n\ndef yn(col):\n    \"\"\"Y/N/yes/no/true/false/1/0/T/F to a real boolean.\"\"\"\n    return (\"CASE WHEN lower(trim(coalesce({c},''))) IN ('y','yes','true','1','t') THEN TRUE \"\n            \"WHEN lower(trim(coalesce({c},''))) IN ('n','no','false','0','f') THEN FALSE \"\n            \"ELSE NULL END\").format(c=col)\n\n\n# --------------------------------------------------------------------------\n# conformance\n# --------------------------------------------------------------------------\n\nCOUNTRY = \"\"\"CASE WHEN upper(trim(coalesce({c},''))) IN\n   ('MY','MYS','MALAYSIA','458') THEN 'MY' ELSE NULL END\"\"\"\n\nTXN_TYPE = \"\"\"CASE\n   WHEN upper(trim(coalesce({c},''))) IN ('DEBIT','DR','D') THEN 'DEBIT'\n   WHEN upper(trim(coalesce({c},''))) IN ('CREDIT','CR','C') THEN 'CREDIT'\n   ELSE NULL END\"\"\"\n\n# 11 stored variants describe 5 real statuses. A worklist filtered on 'PENDING'\n# silently misses the 251,386 customers stored as 'P'. That is the compliance\n# gap this mapping closes.\nKYC = \"\"\"CASE\n   WHEN upper(trim(coalesce({c},''))) IN ('VERIFIED','V') THEN 'VERIFIED'\n   WHEN upper(trim(coalesce({c},''))) IN ('PENDING','P') THEN 'PENDING'\n   WHEN upper(trim(coalesce({c},''))) = 'REJECTED' THEN 'REJECTED'\n   WHEN upper(trim(coalesce({c},''))) = 'EXPIRED' THEN 'EXPIRED'\n   WHEN upper(trim(coalesce({c},''))) IN ('NOT STARTED','INCOMPLETE') THEN 'NOT_STARTED'\n   ELSE NULL END\"\"\"\n\nCHANNEL = \"\"\"CASE\n   WHEN upper(trim(coalesce({c},''))) IN ('MOBILE','MOBILE APP') THEN 'MOBILE'\n   WHEN upper(trim(coalesce({c},''))) IN ('WEB','IB','INTERNET BANKING') THEN 'INTERNET'\n   WHEN upper(trim(coalesce({c},''))) = 'ATM' THEN 'ATM'\n   WHEN upper(trim(coalesce({c},''))) = 'BRANCH' THEN 'BRANCH'\n   WHEN upper(trim(coalesce({c},''))) IN ('CALL CENTRE','CALL CENTER') THEN 'CALL_CENTRE'\n   WHEN upper(trim(coalesce({c},''))) = 'POS' THEN 'POS'\n   WHEN upper(trim(coalesce({c},''))) IN ('API','AGENT') THEN 'OTHER'\n   ELSE NULL END\"\"\"\n\nCARD_BRAND = \"\"\"CASE\n   WHEN upper(trim(coalesce({c},''))) IN ('VISA') THEN 'VISA'\n   WHEN upper(trim(coalesce({c},''))) IN ('MASTERCARD','MC') THEN 'MASTERCARD'\n   WHEN upper(trim(coalesce({c},''))) IN ('AMEX','AMERICAN EXPRESS') THEN 'AMEX'\n   WHEN upper(trim(coalesce({c},''))) = 'MYDEBIT' THEN 'MYDEBIT'\n   WHEN upper(trim(coalesce({c},''))) = 'UNIONPAY' THEN 'UNIONPAY'\n   ELSE NULL END\"\"\"\n\nAPPROVED = \"\"\"CASE WHEN upper(trim(coalesce({c},''))) IN ('00','0','000','APPROVED')\n   THEN TRUE WHEN {n} THEN NULL ELSE FALSE END\"\"\"\n\n\ndef silver_sql(src):\n    \"\"\"Clean + conform. One pass, no joins, so it scales linearly.\"\"\"\n    return \"\"\"\nSELECT\n  _mirror_row_id,\n  {txn_id}                                       AS txn_id,\n  {txn_ts}                                       AS txn_ts,\n  {post_dt}                                      AS posting_date,\n  {val_dt}                                       AS value_date,\n  {txn_type}                                     AS txn_type,\n  {txn_status}                                   AS txn_status,\n  {acct}                                         AS account_no,\n  {acct_type}                                    AS account_type,\n  {acct_cur}                                     AS account_currency,\n  {cust}                                         AS customer_id,\n  {first}                                        AS first_name,\n  {last}                                         AS last_name,\n  {dob}                                          AS date_of_birth,\n  {gender}                                       AS gender,\n  {nationality}                                  AS nationality,\n  {occupation}                                   AS occupation,\n  {city}                                         AS addr_city,\n  {state}                                        AS addr_state,\n  {addr_country}                                 AS addr_country,\n  {kyc}                                          AS kyc_status,\n  {kyc_dt}                                       AS kyc_review_date,\n  {risk}                                         AS risk_rating,\n  {pep}                                          AS pep_flag,\n  {sanctions}                                    AS sanctions_hit,\n  {merch_id}                                     AS merchant_id,\n  {merch}                                        AS merchant_name,\n  {mcc}                                          AS merchant_mcc,\n  {merch_country}                                AS merchant_country,\n  {amt}                                          AS txn_amount,\n  {cur}                                          AS txn_currency,\n  {amt_local}                                    AS amount_local,\n  {fee}                                          AS fee_amount,\n  {card_bin}                                     AS card_bin,\n  {card_brand}                                   AS card_brand,\n  {entry}                                        AS entry_mode,\n  {approved}                                     AS auth_approved,\n  {channel}                                      AS channel,\n  {source}                                       AS source_system\nFROM {src}\n\"\"\".format(\n        src=src,\n        txn_id=clean_text(\"txn_id\"),\n        txn_ts=to_ts(\"txn_datetime\"),\n        post_dt=to_date(\"posting_date\"),\n        val_dt=to_date(\"value_date\"),\n        txn_type=TXN_TYPE.format(c=\"txn_type\"),\n        txn_status=clean_upper(\"txn_status\"),\n        acct=clean_text(\"account_no\"),\n        acct_type=clean_upper(\"account_type\"),\n        acct_cur=clean_upper(\"account_currency\"),\n        cust=clean_text(\"customer_id\"),\n        first=clean_text(\"first_name\"),\n        last=clean_text(\"last_name\"),\n        dob=to_date(\"dob\"),\n        gender=(\"CASE WHEN lower(trim(coalesce(gender,''))) IN ('m','male','1') THEN 'M' \"\n                \"WHEN lower(trim(coalesce(gender,''))) IN ('f','female','2') THEN 'F' ELSE NULL END\"),\n        nationality=COUNTRY.format(c=\"nationality\"),\n        occupation=clean_upper(\"occupation\"),\n        city=clean_upper(\"addr_city\"),\n        state=clean_upper(\"addr_state\"),\n        addr_country=COUNTRY.format(c=\"addr_country\"),\n        kyc=KYC.format(c=\"kyc_status\"),\n        kyc_dt=to_date(\"kyc_review_date\"),\n        risk=(\"CASE WHEN upper(trim(coalesce(risk_rating,''))) IN ('LOW','L','1') THEN 'LOW' \"\n              \"WHEN upper(trim(coalesce(risk_rating,''))) IN ('MEDIUM','MED','M','2') THEN 'MEDIUM' \"\n              \"WHEN upper(trim(coalesce(risk_rating,''))) IN ('HIGH','H','3') THEN 'HIGH' ELSE NULL END\"),\n        pep=yn(\"pep_flag\"),\n        sanctions=yn(\"sanctions_hit\"),\n        merch_id=clean_text(\"merchant_id\"),\n        merch=clean_upper(\"merchant_name\"),\n        mcc=(\"CASE WHEN regexp_matches(trim(coalesce(merchant_mcc,'')), '^[0-9]{4}$') \"\n             \"THEN trim(merchant_mcc) ELSE NULL END\"),\n        merch_country=COUNTRY.format(c=\"merchant_country\"),\n        amt=money(\"txn_amount\"),\n        cur=clean_upper(\"txn_currency\"),\n        amt_local=money(\"amount_local\"),\n        fee=money(\"fee_amount\"),\n        card_bin=(\"CASE WHEN regexp_matches(trim(coalesce(card_bin,'')), '^[0-9]{6}$') \"\n                  \"THEN trim(card_bin) ELSE NULL END\"),\n        card_brand=CARD_BRAND.format(c=\"card_brand\"),\n        entry=clean_upper(\"entry_mode\"),\n        approved=APPROVED.format(c=\"auth_response\", n=\"lower(trim(coalesce(auth_response,''))) IN \" + NULLISH),\n        channel=CHANNEL.format(c=\"channel\"),\n        source=clean_upper(\"source_system\"),\n    )\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--source\", choices=[\"postgres\", \"delta\"], default=\"postgres\")\n    p.add_argument(\"--path\", default=\"/lakehouse/default/Tables/raw_txn\")\n    p.add_argument(\"--limit\", type=int, default=0)\n    p.add_argument(\"--memory-limit\", default=\"10GB\")\n    p.add_argument(\"--host\", default=\"localhost\")\n    p.add_argument(\"--dbname\", default=\"postgres\")\n    p.add_argument(\"--user\", default=\"postgres\")\n    args = p.parse_args()\n\n    con = duckdb.connect()\n    con.execute(\"SET memory_limit='%s'\" % args.memory_limit)\n    con.execute(\"SET preserve_insertion_order=false\")\n    con.execute(\"SET TimeZone='UTC'\")   # results must not depend on where this runs\n\n    if args.source == \"postgres\":\n        con.execute(\"INSTALL postgres\"); con.execute(\"LOAD postgres\")\n        con.execute(\"ATTACH 'host=%s dbname=%s user=%s' AS pg (TYPE postgres, READ_ONLY)\"\n                    % (args.host, args.dbname, args.user))\n        src = \"pg.landing.raw_txn\"\n    else:\n        con.execute(\"INSTALL delta\"); con.execute(\"LOAD delta\")\n        src = \"delta_scan('%s')\" % args.path\n\n    if args.limit:\n        # Deterministic slice. LIMIT without ORDER BY is nondeterministic over a\n        # parallel scan: two runs returned samples differing by 23,008 duplicate\n        # txn_ids, which made every measurement irreproducible. Ranging over the\n        # surrogate key gives the same rows every time.\n        src = \"(SELECT * FROM %s WHERE _mirror_row_id <= %d)\" % (src, args.limit)\n\n    t0 = time.time()\n    con.execute(\"CREATE OR REPLACE TABLE silver AS \" + silver_sql(src))\n    n = con.execute(\"SELECT count(*) FROM silver\").fetchone()[0]\n    print(\"silver built: {:,} rows in {:.1f}s\".format(n, time.time() - t0))\n\n    print(\"\\nparse recovery (share of non-null rows that now carry a real value)\")\n    checks = [\n        (\"posting_date\", \"posting_date IS NOT NULL\"),\n        (\"txn_ts\", \"txn_ts IS NOT NULL\"),\n        (\"txn_amount\", \"txn_amount IS NOT NULL\"),\n        (\"kyc_status\", \"kyc_status IS NOT NULL\"),\n        (\"channel\", \"channel IS NOT NULL\"),\n        (\"addr_country\", \"addr_country IS NOT NULL\"),\n        (\"card_brand\", \"card_brand IS NOT NULL\"),\n        (\"auth_approved\", \"auth_approved IS NOT NULL\"),\n    ]\n    for label, pred in checks:\n        got = con.execute(\"SELECT round(100.0*sum(CASE WHEN %s THEN 1 ELSE 0 END)/count(*),1) FROM silver\"\n                          % pred).fetchone()[0]\n        print(\"  %-16s %5s%%\" % (label, got))\n\n    print(\"\\nconformance: distinct values after mapping\")\n    for c in (\"addr_country\", \"txn_type\", \"kyc_status\", \"channel\", \"card_brand\"):\n        d = con.execute(\"SELECT count(DISTINCT %s) FROM silver\" % c).fetchone()[0]\n        print(\"  %-16s %d\" % (c, d))\n\n    print(\"\\nduplicates\")\n    d = con.execute(\"SELECT count(*) FROM (SELECT txn_id FROM silver WHERE txn_id IS NOT NULL \"\n                    \"GROUP BY txn_id HAVING count(*)>1)\").fetchone()[0]\n    print(\"  txn_id appearing more than once: {:,}\".format(d))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
 "rules.py": "\"\"\"\nPhase 2 risk layer: the nine rules from APP_DESIGN.md, over conformed silver.\n\n  python rules.py --limit 2000000\n\nWhy these rules exist at all\n---------------------------\nThe source `fraud_score` column is uncorrelated with everything. Measured on the\nraw table it sits between 29.9 and 31.2 across every single channel, so a chart\nbuilt on it shows fraud spread perfectly evenly and is worthless.\n\nThese rules are computed instead, from columns that genuinely vary. The score is\ntheir weighted sum, so it has a real distribution and correlates with its own\ndrivers by construction.\n\nA rule that fires on most rows carries no information, however sound it sounds.\nThis module measures each rule's fire rate and lift so that can be judged rather\nthan assumed, and reports which rules should be dropped.\n\"\"\"\n\nimport argparse\nimport os\nimport sys\nimport time\n\nimport duckdb\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom transform import silver_sql  # noqa: E402\n\n# Weight is how much a firing rule adds to the score. Rare, specific signals are\n# worth more than common ones; these are starting weights, recalibrated below\n# against the fire rates this module measures.\n# Four of the original nine were measured against the real data and dropped or\n# recalibrated. A rule that fires on four rows in five cannot rank anything,\n# however sound it sounds in a policy document.\n#   R01 cross_border  fired on 80.22%. Currency is uniform in this source, so\n#                     the rule carries no information. DROPPED.\n#   R06 new_merchant  fired on 85.88%. Most customer/merchant pairs here are\n#                     unique, so \"first time\" is the normal case. Restricted to\n#                     customers with enough history for it to mean something.\n#   R05 velocity      fired on 0.01% at three per account-hour. Lowered to two.\n#   R08 round_amount  fired on 0.00%. Amounts come from a continuous\n#                     distribution and are essentially never exact hundreds.\n#                     DROPPED.\nRULES = [\n    (\"R02\", \"card_not_present\",  \"entry_mode IN ('ECOM','E-COMMERCE','KEYED','MANUAL')\", 25),\n    (\"R03\", \"amount_anomaly\",    \"amt_pct_in_customer >= 0.95\", 30),\n    (\"R04\", \"odd_hour\",          \"txn_hour BETWEEN 1 AND 5\", 15),\n    (\"R05\", \"velocity\",          \"acct_txns_same_hour >= 2\", 30),\n    (\"R06\", \"new_merchant\",      \"first_seen_merchant AND cust_txn_count >= 20\", 10),\n    (\"R07\", \"high_risk_mcc\",     \"merchant_mcc IN ('6011','4829','7995')\", 25),\n    (\"R09\", \"declined\",          \"auth_approved = FALSE\", 10),\n]\n\n\ndef enriched_sql(silver):\n    \"\"\"Window features the rules need. One pass over the silver table.\"\"\"\n    return \"\"\"\nSELECT *,\n  hour(txn_ts) AS txn_hour,\n  -- where this amount sits inside this customer's own spending history.\n  -- NULL for a customer with a single transaction, which correctly means\n  -- \"no basis to call it anomalous\" rather than \"not anomalous\".\n  CASE WHEN customer_id IS NULL OR txn_amount IS NULL THEN NULL\n       WHEN count(*) OVER (PARTITION BY customer_id) < 5 THEN NULL\n       ELSE percent_rank() OVER (PARTITION BY customer_id ORDER BY txn_amount)\n  END AS amt_pct_in_customer,\n  -- transactions on the same account inside the same clock hour\n  count(*) OVER (PARTITION BY account_no, date_trunc('hour', txn_ts))\n       AS acct_txns_same_hour,\n  -- how much history this customer has, so \"first time at this merchant\" is\n  -- only treated as a signal where there was a pattern to depart from\n  count(*) OVER (PARTITION BY customer_id) AS cust_txn_count,\n  -- first time this customer transacted at this merchant. Exact duplicate rows\n  -- tie on txn_ts, so the tiebreak must agree with dedup (lowest _mirror_row_id\n  -- survives); otherwise the group's only firing can land on the copy that is\n  -- dropped, and R06 varied by hundreds between runs. BUILD_LOG section 40.\n  CASE WHEN customer_id IS NULL OR merchant_id IS NULL THEN FALSE\n       ELSE row_number() OVER (PARTITION BY customer_id, merchant_id\n                               ORDER BY txn_ts, _mirror_row_id) = 1\n  END AS first_seen_merchant\nFROM ({s})\n\"\"\".format(s=silver)\n\n\ndef scored_sql(enriched):\n    flags = \",\\n  \".join(\n        \"CASE WHEN {c} THEN TRUE ELSE FALSE END AS {n}\".format(c=cond, n=name)\n        for _, name, cond, _ in RULES)\n    score = \" + \".join(\n        \"CASE WHEN {c} THEN {w} ELSE 0 END\".format(c=cond, w=w)\n        for _, _, cond, w in RULES)\n    fired = \" + \".join(\n        \"CASE WHEN {c} THEN 1 ELSE 0 END\".format(c=cond) for _, _, cond, _ in RULES)\n    return \"\"\"\nSELECT *,\n  {flags},\n  ({score}) AS risk_score,\n  ({fired}) AS rules_fired\nFROM ({e})\n\"\"\".format(flags=flags, score=score, fired=fired, e=enriched)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--limit\", type=int, default=2000000)\n    p.add_argument(\"--memory-limit\", default=\"10GB\")\n    p.add_argument(\"--host\", default=\"localhost\")\n    p.add_argument(\"--dbname\", default=\"postgres\")\n    p.add_argument(\"--user\", default=\"postgres\")\n    args = p.parse_args()\n\n    con = duckdb.connect()\n    con.execute(\"SET memory_limit='%s'\" % args.memory_limit)\n    con.execute(\"SET preserve_insertion_order=false\")\n    con.execute(\"SET TimeZone='UTC'\")   # results must not depend on where this runs\n    con.execute(\"INSTALL postgres\"); con.execute(\"LOAD postgres\")\n    con.execute(\"ATTACH 'host=%s dbname=%s user=%s' AS pg (TYPE postgres, READ_ONLY)\"\n                % (args.host, args.dbname, args.user))\n\n    src = \"pg.landing.raw_txn\"\n    if args.limit:\n        # Deterministic slice. LIMIT without ORDER BY is nondeterministic over a\n        # parallel scan: two runs returned samples differing by 23,008 duplicate\n        # txn_ids, which made every measurement irreproducible. Ranging over the\n        # surrogate key gives the same rows every time.\n        src = \"(SELECT * FROM %s WHERE _mirror_row_id <= %d)\" % (src, args.limit)\n\n    t0 = time.time()\n    con.execute(\"CREATE OR REPLACE TABLE scored AS \"\n                + scored_sql(enriched_sql(silver_sql(src))))\n    n = con.execute(\"SELECT count(*) FROM scored\").fetchone()[0]\n    print(\"scored: {:,} rows in {:.1f}s\".format(n, time.time() - t0))\n\n    print(\"\\nRULE FIRE RATES\")\n    print(\"  a rule firing on most rows cannot discriminate, however sensible it sounds\")\n    rates = {}\n    for rid, name, _, w in RULES:\n        r = con.execute(\"SELECT round(100.0*sum(CASE WHEN {n} THEN 1 ELSE 0 END)/count(*),2) \"\n                        \"FROM scored\".format(n=name)).fetchone()[0]\n        rates[rid] = r\n        verdict = \"too common\" if r > 50 else (\"too rare\" if r < 0.5 else \"usable\")\n        print(\"  %-4s %-18s %7s%%  weight %-3s  %s\" % (rid, name, r, w, verdict))\n\n    print(\"\\nRISK SCORE DISTRIBUTION\")\n    rows = con.execute(\"\"\"\n      SELECT CASE WHEN risk_score < 20 THEN '  0-19' WHEN risk_score < 40 THEN ' 20-39'\n                  WHEN risk_score < 60 THEN ' 40-59' WHEN risk_score < 80 THEN ' 60-79'\n                  ELSE '80+' END AS band,\n             count(*) n, round(100.0*count(*)/sum(count(*)) OVER (),1) pct\n      FROM scored GROUP BY 1 ORDER BY 1\"\"\").fetchall()\n    for b, c, pct in rows:\n        bar = \"#\" * int(round(pct / 2))\n        print(\"  %-6s %9s  %5s%%  %s\" % (b, \"{:,}\".format(c), pct, bar))\n\n    stats = con.execute(\"SELECT min(risk_score), round(avg(risk_score),1), \"\n                        \"max(risk_score), round(stddev(risk_score),1) FROM scored\").fetchone()\n    print(\"  min %s  mean %s  max %s  stddev %s\" % stats)\n\n    print(\"\\nDOES THE SCORE CORRELATE WITH ITS OWN DRIVERS?\")\n    print(\"  share of each rule firing, inside the top decile vs the bottom decile\")\n    for rid, name, _, _ in RULES:\n        hi, lo = con.execute(\"\"\"\n          SELECT round(100.0*avg(CASE WHEN {n} THEN 1.0 ELSE 0 END) FILTER (WHERE risk_score >= h),1),\n                 round(100.0*avg(CASE WHEN {n} THEN 1.0 ELSE 0 END) FILTER (WHERE risk_score <= l),1)\n          FROM scored, (SELECT quantile_cont(risk_score,0.9) h, quantile_cont(risk_score,0.1) l FROM scored)\n        \"\"\".format(n=name)).fetchone()\n        print(\"  %-4s %-18s top %6s%%   bottom %6s%%\" % (rid, name, hi, lo))\n\n    print(\"\\nDISCRIMINATION: DERIVED SCORE vs THE SOURCE fraud_score COLUMN\")\n    print(\"  Raw spread is not comparable, the two sit on different scales.\")\n    print(\"  What matters is whether a score separates groups at all, so this\")\n    print(\"  measures how far the per-group means move, relative to the scale.\")\n    con.execute(\"\"\"CREATE OR REPLACE TABLE src_cmp AS\n      SELECT upper(trim(entry_mode)) AS entry_mode,\n             TRY_CAST(regexp_replace(coalesce(fraud_score,''),'[^0-9.]','','g') AS DOUBLE) AS fs\n      FROM pg.landing.raw_txn LIMIT %d\"\"\" % args.limit)\n    for label, expr, tbl in [(\"derived risk_score\", \"risk_score\", \"scored\"),\n                             (\"source fraud_score\", \"fs\", \"src_cmp\")]:\n        sd, mean = con.execute(\"\"\"\n          SELECT round(stddev(m),3), round(avg(m),3) FROM (\n            SELECT avg({e}) m FROM {t}\n            WHERE entry_mode IS NOT NULL AND trim(entry_mode) <> ''\n            GROUP BY entry_mode HAVING count(*) > 1000)\"\"\".format(e=expr, t=tbl)).fetchone()\n        sd = sd or 0.0; mean = mean or 0.0\n        cv = round(100.0 * sd / mean, 2) if mean else 0\n        print(\"    %-18s group means vary by %8s around %9s  =  %5s%% of scale\"\n              % (label, sd, mean, cv))\n    print(\"  A score whose group means barely move cannot rank anything.\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
 "star.py": "\"\"\"\nPhase 2 gold layer: dimensions and facts for the Financial Crime Ops model.\n\n  python star.py --limit 2000000            # build and profile locally\n  python star.py --limit 2000000 --out D:/duckdb-fabric-data/gold\n\nGrain decisions, which are the part worth arguing about\n-------------------------------------------------------\n* `fact_transaction` is one row per `_mirror_row_id`. That is the surrogate key\n  added so PostgreSQL would emit UPDATE and DELETE, and it is also the join key\n  the write-back layer uses to tie an analyst's decision to a transaction. Using\n  anything else would break that chain.\n* `fact_alert` is one row per rule firing, not per transaction. A transaction\n  that trips three rules produces three alert rows. That is what makes \"which\n  rules earn their keep\" answerable without unpivoting at query time.\n* Duplicates are resolved here, not earlier. The 593,209 exact duplicate rows\n  are real source behaviour and belong in the landing layer untouched; the fact\n  table is where a defensible single version is chosen.\n\"\"\"\n\nimport argparse\nimport os\nimport sys\nimport time\n\nimport duckdb\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom transform import silver_sql          # noqa: E402\nfrom rules import RULES, enriched_sql, scored_sql  # noqa: E402\n\n\ndef export_source(con, table):\n    \"\"\"\n    What COPY reads when a gold table is exported to parquet.\n\n    Naive TIMESTAMP columns are cast to TIMESTAMPTZ. The session is pinned to\n    UTC, so the instant equals the naive value read as UTC, and the parquet\n    carries isAdjustedToUTC=true. Spark reads that as an ordinary timestamp; a\n    naive column it reads as TIMESTAMP_NTZ, which the existing Delta tables do\n    not accept and Direct Lake does not serve. BUILD_LOG section 40.\n    \"\"\"\n    cols = [r[0] for r in con.execute(\n        \"SELECT column_name FROM information_schema.columns \"\n        \"WHERE table_name = ? AND data_type = 'TIMESTAMP' ORDER BY ordinal_position\",\n        [table]).fetchall()]\n    if not cols:\n        return table\n    rep = \", \".join(\"CAST(%s AS TIMESTAMPTZ) AS %s\" % (c, c) for c in cols)\n    return \"(SELECT * REPLACE (%s) FROM %s)\" % (rep, table)\n\n\ndef build(con, src):\n    con.execute(\"CREATE OR REPLACE TABLE scored AS \"\n                + scored_sql(enriched_sql(silver_sql(src))))\n\n    # ---- deduplicate -----------------------------------------------------\n    # Exact duplicates share every business column. Keep the lowest\n    # _mirror_row_id as the survivor: it is stable, reproducible, and points at\n    # the first time the row was seen.\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE deduped AS\n      SELECT * EXCLUDE (rn) FROM (\n        SELECT *, row_number() OVER (\n          PARTITION BY txn_id, txn_amount, merchant_name, txn_ts\n          ORDER BY _mirror_row_id) AS rn\n        FROM scored)\n      WHERE rn = 1\n    \"\"\")\n\n    # ---- dimensions ------------------------------------------------------\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_date AS\n      SELECT row_number() OVER (ORDER BY d) AS date_key, d AS full_date,\n             year(d) AS yr, quarter(d) AS qtr, month(d) AS mth,\n             strftime(d,'%b') AS month_name, day(d) AS day_of_month,\n             dayofweek(d) AS day_of_week, strftime(d,'%a') AS day_name,\n             CASE WHEN dayofweek(d) IN (0,6) THEN TRUE ELSE FALSE END AS is_weekend\n      FROM (SELECT DISTINCT CAST(txn_ts AS DATE) d FROM deduped WHERE txn_ts IS NOT NULL)\n      ORDER BY d\n    \"\"\")\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_time AS\n      SELECT h AS time_key, h AS hour_of_day,\n             printf('%02d:00', h) AS hour_label,\n             CASE WHEN h BETWEEN 1 AND 5 THEN 'Overnight'\n                  WHEN h BETWEEN 6 AND 11 THEN 'Morning'\n                  WHEN h BETWEEN 12 AND 17 THEN 'Afternoon'\n                  ELSE 'Evening' END AS day_part,\n             CASE WHEN h BETWEEN 1 AND 5 THEN TRUE ELSE FALSE END AS is_odd_hour\n      FROM range(0,24) t(h)\n    \"\"\")\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_customer AS\n      SELECT row_number() OVER (ORDER BY customer_id) AS customer_key, *\n      FROM (\n        SELECT customer_id,\n               any_value(first_name) AS first_name,\n               any_value(last_name)  AS last_name,\n               any_value(gender)     AS gender,\n               any_value(date_of_birth) AS date_of_birth,\n               any_value(occupation) AS occupation,\n               any_value(addr_city)  AS addr_city,\n               any_value(addr_state) AS addr_state,\n               any_value(nationality) AS nationality,\n               -- worst-case wins: a customer flagged once stays flagged\n               max(CASE WHEN kyc_status='VERIFIED' THEN 0 WHEN kyc_status IS NULL THEN 1\n                        ELSE 2 END) AS kyc_rank,\n               max(CASE WHEN pep_flag THEN 1 ELSE 0 END)::BOOLEAN AS ever_pep,\n               max(CASE WHEN sanctions_hit THEN 1 ELSE 0 END)::BOOLEAN AS ever_sanctioned,\n               max(CASE WHEN risk_rating='HIGH' THEN 3 WHEN risk_rating='MEDIUM' THEN 2\n                        WHEN risk_rating='LOW' THEN 1 ELSE 0 END) AS risk_rank,\n               count(*) AS txn_count\n        FROM deduped WHERE customer_id IS NOT NULL GROUP BY customer_id)\n    \"\"\")\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_merchant AS\n      SELECT row_number() OVER (ORDER BY merchant_id) AS merchant_key, *,\n             CASE WHEN merchant_mcc IN ('6011','4829','7995') THEN TRUE ELSE FALSE END\n               AS is_high_risk_mcc\n      FROM (\n        SELECT merchant_id,\n               any_value(merchant_name) AS merchant_name,\n               any_value(merchant_mcc)  AS merchant_mcc,\n               any_value(merchant_country) AS merchant_country,\n               count(*) AS txn_count\n        FROM deduped WHERE merchant_id IS NOT NULL GROUP BY merchant_id)\n    \"\"\")\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_channel AS\n      SELECT row_number() OVER (ORDER BY channel, entry_mode) AS channel_key,\n             channel, entry_mode,\n             CASE WHEN entry_mode IN ('ECOM','E-COMMERCE','KEYED','MANUAL')\n                  THEN TRUE ELSE FALSE END AS is_card_not_present\n      FROM (SELECT DISTINCT channel, entry_mode FROM deduped\n            WHERE channel IS NOT NULL OR entry_mode IS NOT NULL)\n    \"\"\")\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_card AS\n      SELECT row_number() OVER (ORDER BY card_bin, card_brand) AS card_key,\n             card_bin, card_brand\n      FROM (SELECT DISTINCT card_bin, card_brand FROM deduped\n            WHERE card_bin IS NOT NULL OR card_brand IS NOT NULL)\n    \"\"\")\n    rule_rows = \",\".join(\"('%s','%s',%d)\" % (rid, name, w) for rid, name, _, w in RULES)\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE dim_risk_rule AS\n      SELECT * FROM (VALUES %s) AS t(rule_id, rule_name, weight)\n    \"\"\" % rule_rows)\n\n\n    # ---- unknown members -------------------------------------------------\n    # A NULL foreign key silently drops the row out of any aggregate that\n    # filters on that dimension, so 14% of transactions would disappear from a\n    # merchant-sliced total without anyone noticing. Every dimension gets an\n    # explicit Unknown member at key -1 and the fact coalesces to it.\n    con.execute(\"INSERT INTO dim_customer SELECT -1,'UNKNOWN',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,1,FALSE,FALSE,0,0\")\n    con.execute(\"INSERT INTO dim_merchant SELECT -1,'UNKNOWN','Unknown merchant',NULL,NULL,0,FALSE\")\n    con.execute(\"INSERT INTO dim_channel  SELECT -1,'UNKNOWN','UNKNOWN',FALSE\")\n    con.execute(\"INSERT INTO dim_card     SELECT -1,'UNKNOWN','UNKNOWN'\")\n    con.execute(\"INSERT INTO dim_date     SELECT -1,NULL,NULL,NULL,NULL,'Unknown',NULL,NULL,'Unknown',FALSE\")\n    con.execute(\"INSERT INTO dim_time     SELECT -1,-1,'Unknown','Unknown',FALSE\")\n\n    # ---- facts -----------------------------------------------------------\n    con.execute(\"\"\"\n      CREATE OR REPLACE TABLE fact_transaction AS\n      SELECT\n        d._mirror_row_id,\n        d.txn_id,\n        coalesce(dd.date_key,   -1) AS date_key,\n        coalesce(dt.time_key,   -1) AS time_key,\n        coalesce(c.customer_key,-1) AS customer_key,\n        coalesce(m.merchant_key,-1) AS merchant_key,\n        coalesce(ch.channel_key,-1) AS channel_key,\n        coalesce(cd.card_key,   -1) AS card_key,\n        d.txn_ts, d.txn_type, d.txn_status,\n        d.txn_amount, d.txn_currency, d.amount_local, d.fee_amount,\n        d.auth_approved, d.risk_score, d.rules_fired,\n        CASE WHEN d.risk_score >= 60 THEN 'HIGH'\n             WHEN d.risk_score >= 30 THEN 'MEDIUM' ELSE 'LOW' END AS risk_band\n      FROM deduped d\n      LEFT JOIN dim_date     dd ON dd.full_date = CAST(d.txn_ts AS DATE)\n      LEFT JOIN dim_time     dt ON dt.time_key  = hour(d.txn_ts)\n      LEFT JOIN dim_customer c  ON c.customer_id = d.customer_id\n      LEFT JOIN dim_merchant m  ON m.merchant_id = d.merchant_id\n      LEFT JOIN dim_channel  ch ON ch.channel IS NOT DISTINCT FROM d.channel\n                               AND ch.entry_mode IS NOT DISTINCT FROM d.entry_mode\n      LEFT JOIN dim_card     cd ON cd.card_bin IS NOT DISTINCT FROM d.card_bin\n                               AND cd.card_brand IS NOT DISTINCT FROM d.card_brand\n    \"\"\")\n\n    # one row per rule firing, so rule effectiveness is a group-by not an unpivot\n    union = \"\\n UNION ALL \".join(\n        \"SELECT _mirror_row_id, '%s' AS rule_id, %d AS weight FROM deduped WHERE %s\"\n        % (rid, w, name) for rid, name, _, w in RULES)\n    # No surrogate key here. (_mirror_row_id, rule_id) is already unique by\n    # construction, and minting an ordered alert_key meant a global sort over\n    # ~48,000,000 rows, which exhausted 38 GiB of spill space and killed the\n    # build. A surrogate that costs a full sort and identifies nothing the\n    # natural key does not is not worth having.\n    con.execute(\"CREATE OR REPLACE TABLE fact_alert AS SELECT * FROM (%s)\" % union)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--limit\", type=int, default=2000000)\n    p.add_argument(\"--out\", default=\"\")\n    p.add_argument(\"--memory-limit\", default=\"10GB\")\n    p.add_argument(\"--temp-dir\", default=\"D:/duckdb-fabric-data/tmp\")\n    p.add_argument(\"--host\", default=\"localhost\")\n    p.add_argument(\"--dbname\", default=\"postgres\")\n    p.add_argument(\"--user\", default=\"postgres\")\n    args = p.parse_args()\n\n    con = duckdb.connect()\n    con.execute(\"SET memory_limit='%s'\" % args.memory_limit)\n    con.execute(\"SET preserve_insertion_order=false\")\n    con.execute(\"SET TimeZone='UTC'\")   # results must not depend on where this runs\n    # Spill to D:. The default temp directory follows the database file, which\n    # here means C: with 43 GB free against D:'s 479 GB.\n    con.execute(\"SET temp_directory='%s'\" % args.temp_dir)\n    con.execute(\"INSTALL postgres\"); con.execute(\"LOAD postgres\")\n    con.execute(\"ATTACH 'host=%s dbname=%s user=%s' AS pg (TYPE postgres, READ_ONLY)\"\n                % (args.host, args.dbname, args.user))\n    src = \"pg.landing.raw_txn\"\n    if args.limit:\n        # Deterministic slice. LIMIT without ORDER BY is nondeterministic over a\n        # parallel scan: two runs returned samples differing by 23,008 duplicate\n        # txn_ids, which made every measurement irreproducible. Ranging over the\n        # surrogate key gives the same rows every time.\n        src = \"(SELECT * FROM %s WHERE _mirror_row_id <= %d)\" % (src, args.limit)\n\n    t0 = time.time()\n    build(con, src)\n    print(\"star built in %.1f s\" % (time.time() - t0))\n\n    print(\"\\nTABLE SIZES\")\n    for t in (\"fact_transaction\", \"fact_alert\", \"dim_customer\", \"dim_merchant\",\n              \"dim_channel\", \"dim_card\", \"dim_date\", \"dim_time\", \"dim_risk_rule\"):\n        n = con.execute(\"SELECT count(*) FROM %s\" % t).fetchone()[0]\n        print(\"  %-18s %12s\" % (t, \"{:,}\".format(n)))\n\n    raw, ded = con.execute(\"SELECT (SELECT count(*) FROM scored), \"\n                           \"(SELECT count(*) FROM deduped)\").fetchone()\n    print(\"\\nDEDUPLICATION\\n  scored {:,} -> deduped {:,}, removed {:,} ({:.2f}%)\".format(\n        raw, ded, raw - ded, 100.0 * (raw - ded) / raw))\n\n    print(\"\\nREFERENTIAL INTEGRITY (unmatched dimension keys in the fact)\")\n    for k in (\"date_key\", \"time_key\", \"customer_key\", \"merchant_key\", \"channel_key\", \"card_key\"):\n        miss = con.execute(\"SELECT round(100.0*sum(CASE WHEN %s = -1 THEN 1 ELSE 0 END)\"\n                           \"/count(*),2) FROM fact_transaction\" % k).fetchone()[0]\n        nulls = con.execute(\"SELECT sum(CASE WHEN %s IS NULL THEN 1 ELSE 0 END) \"\n                            \"FROM fact_transaction\" % k).fetchone()[0]\n        print(\"  %-14s %6s%% unknown, %s null FKs\" % (k, miss, nulls))\n\n    print(\"\\nRISK BAND MIX\")\n    for b, n, pct in con.execute(\"\"\"\n        SELECT risk_band, count(*), round(100.0*count(*)/sum(count(*)) OVER (),1)\n        FROM fact_transaction GROUP BY 1 ORDER BY 2 DESC\"\"\").fetchall():\n        print(\"  %-7s %10s  %5s%%\" % (b, \"{:,}\".format(n), pct))\n\n    print(\"\\nRULE EFFECTIVENESS (from fact_alert, a group-by not an unpivot)\")\n    for rid, name, n, pct in con.execute(\"\"\"\n        SELECT a.rule_id, r.rule_name, count(*),\n               round(100.0*count(*)/(SELECT count(*) FROM fact_transaction),2)\n        FROM fact_alert a JOIN dim_risk_rule r USING (rule_id)\n        GROUP BY 1,2 ORDER BY 3 DESC\"\"\").fetchall():\n        print(\"  %-5s %-18s %10s alerts  %6s%% of txns\" % (rid, name, \"{:,}\".format(n), pct))\n\n    if args.out:\n        os.makedirs(args.out, exist_ok=True)\n        for t in (\"fact_transaction\", \"fact_alert\", \"dim_customer\", \"dim_merchant\",\n                  \"dim_channel\", \"dim_card\", \"dim_date\", \"dim_time\", \"dim_risk_rule\"):\n            con.execute(\"COPY %s TO '%s/%s.parquet' (FORMAT PARQUET, COMPRESSION zstd)\"\n                        % (export_source(con, t), args.out.replace('\\\\', '/'), t))\n        print(\"\\ngold written to %s\" % args.out)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
}

SCRIPTS = "/tmp/fc_scripts"
os.makedirs(SCRIPTS, exist_ok=True)
for name, src in SOURCES.items():
    with open(os.path.join(SCRIPTS, name), "w", encoding="utf-8") as fh:
        fh.write(src)
sys.path.insert(0, SCRIPTS)

from transform import silver_sql
from rules import RULES, enriched_sql, scored_sql
from star import build, export_source

print("rules active:", [r[0] for r in RULES])

### Read the mirror through the shortcut

The shortcut resolves to the mirrored database's Delta table. It is read-only,
which is correct: nothing here should be able to write back into the mirror.
DuckDB's memory limit is set from the RAM the node actually has, not from a
constant, so the notebook behaves sensibly whatever size it lands on.

In [ ]:
import duckdb

con = duckdb.connect()
limit_gb = max(4, int((ENV["ram_gb"] or 16) * 0.70))
con.execute(f"SET memory_limit='{limit_gb}GB'")
con.execute(f"SET threads={ENV['cpu_count']}")
con.execute("SET preserve_insertion_order=false")
con.execute("SET TimeZone='UTC'")   # results must not depend on where this runs
os.makedirs("/tmp/duckdb", exist_ok=True)
con.execute("SET temp_directory='/tmp/duckdb'")
con.execute("INSTALL delta"); con.execute("LOAD delta")

SRC = "delta_scan('/lakehouse/default/Tables/raw_txn')"
TIMING = {}

t0 = time.time()
SOURCE_ROWS = con.execute(f"SELECT count(*) FROM {SRC}").fetchone()[0]
TIMING["count_source_s"] = round(time.time() - t0, 1)
print(f"duckdb {duckdb.__version__}  memory_limit {limit_gb}GB  threads {ENV['cpu_count']}")
print(f"source rows: {SOURCE_ROWS:,}  ({TIMING['count_source_s']}s)")

### Build the star

One call. Cleaning, conforming, the seven risk rules, scoring, deduplication on
`_mirror_row_id`, and the two facts plus seven dimensions with explicit Unknown
members at key -1.

In [ ]:
t0 = time.time()
build(con, SRC)
TIMING["build_s"] = round(time.time() - t0, 1)
print(f"star built in {TIMING['build_s']/60:.1f} min")

COUNTS = {}
for t in ["fact_transaction", "fact_alert", "dim_customer", "dim_merchant", "dim_channel", "dim_card", "dim_date", "dim_time", "dim_risk_rule"]:
    COUNTS[t] = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"  {t:<18} {COUNTS[t]:>12,}")

# Per-rule firings, so two runs can be compared rule by rule and not just in
# total. A total can hide two errors that cancel.
RULE_FIRINGS = dict(con.execute(
    "SELECT rule_id, count(*) FROM fact_alert GROUP BY 1 ORDER BY 1").fetchall())
print("rule firings:", RULE_FIRINGS)

# Dedup reconciliation. The key is (txn_id, txn_amount, merchant_name, txn_ts),
# so a txn_id that survives more than once is either a genuine business
# difference between its copies or a defect, and it is named here rather than
# averaged away. NULL txn_ids are counted separately: they group by the other
# three columns and are expected.
RECON = {
    "scored_rows": con.execute("SELECT count(*) FROM scored").fetchone()[0],
    "deduped_rows": con.execute("SELECT count(*) FROM deduped").fetchone()[0],
    "distinct_txn_id_scored": con.execute(
        "SELECT count(DISTINCT txn_id) FROM scored").fetchone()[0],
    "null_txn_id_deduped": con.execute(
        "SELECT count(*) FROM deduped WHERE txn_id IS NULL").fetchone()[0],
}
RECON["removed"] = RECON["scored_rows"] - RECON["deduped_rows"]
RECON["multi_survivor_txn_ids"] = con.execute('''
    SELECT count(*) FROM (SELECT txn_id FROM deduped WHERE txn_id IS NOT NULL
                          GROUP BY txn_id HAVING count(*) > 1)''').fetchone()[0]
RECON["multi_survivor_samples"] = []
for (txn_id,) in con.execute('''
        SELECT txn_id FROM deduped WHERE txn_id IS NOT NULL
        GROUP BY txn_id HAVING count(*) > 1 ORDER BY txn_id LIMIT 10''').fetchall():
    rows = con.execute(
        "SELECT _mirror_row_id, txn_amount, merchant_name, txn_ts FROM deduped "
        "WHERE txn_id = ? ORDER BY 1", [txn_id]).fetchall()
    RECON["multi_survivor_samples"].append(
        {"txn_id": txn_id, "rows": [[str(x) for x in r] for r in rows]})
print(json.dumps(RECON, indent=2))

### Write gold as parquet

Overwrites `Files/gold`. The build is deterministic: timestamps are parsed as
naive UTC regardless of the session's timezone and every window tiebreak agrees
with dedup (BUILD_LOG section 40), so two runs over the same source produce the
same counts, rule by rule. Timestamp columns are exported as instants
(TIMESTAMPTZ, `isAdjustedToUTC=true`) because Spark reads a naive column as
TIMESTAMP_NTZ, which the Delta tables and Direct Lake do not take. The V-Order
Delta write is deliberately left to Spark
in `02_vorder_write`: DuckDB cannot produce V-Order, and V-Order is what makes
Direct Lake fast.

In [ ]:
GOLD = "/lakehouse/default/Files/gold"
os.makedirs(GOLD, exist_ok=True)

t0 = time.time()
SIZES_MB = {}
for t in ["fact_transaction", "fact_alert", "dim_customer", "dim_merchant", "dim_channel", "dim_card", "dim_date", "dim_time", "dim_risk_rule"]:
    path = f"{GOLD}/{t}.parquet"
    con.execute(f"COPY {export_source(con, t)} TO '{path}' (FORMAT PARQUET, COMPRESSION zstd)")
    SIZES_MB[t] = round(os.path.getsize(path) / 1e6, 1)
TIMING["write_s"] = round(time.time() - t0, 1)
TIMING["total_s"] = round(sum(TIMING.values()), 1)
print(f"gold parquet written in {TIMING['write_s']:.0f}s, "
      f"{sum(SIZES_MB.values())/1000:.2f} GB")

### Leave evidence

A CLI-triggered run has no visible cell output, so the summary is written to
`Files/build_gold.json` and returned as the notebook's exit value. The README
notebook reads this file, which is how its cost figures stay measured rather
than typed.

In [ ]:
SUMMARY = {
    "ran_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "kernel": "python",
    "vcores_requested": 8,
    "env": ENV,
    "duckdb_version": duckdb.__version__,
    "memory_limit_gb": limit_gb,
    "source_rows": SOURCE_ROWS,
    "timing_s": TIMING,
    "counts": COUNTS,
    "rule_firings": RULE_FIRINGS,
    "recon": RECON,
    "gold_parquet_mb": SIZES_MB,
    "cu": 8 / 2,
    "cu_seconds": round(8 / 2 * TIMING["total_s"], 1),
}
with open("/lakehouse/default/Files/build_gold.json", "w") as fh:
    json.dump(SUMMARY, fh, indent=2)
print(json.dumps(SUMMARY, indent=2))

try:
    import notebookutils
    notebookutils.notebook.exit(json.dumps(SUMMARY))
except Exception as e:
    print("exit value not set:", e)